# Latent Trajectory Dynamics: A Pedagogical Walkthrough

This notebook reproduces the explanation below verbatim, then evaluates every quantity step-by-step for a toy example with:

- `4` groups,
- `2` pairs per group,
- `3` seeds per pair,
- latent size `N = 4`.

It ends with visualizations that make the averaging structure easier to see.

## Verbatim Walkthrough

Yes. The math is consistent, and the two divisions are doing different jobs.

**Key idea**
- $1/N$ inside $d_t^{(p,s)}$ turns a squared Euclidean distance for one pair-seed record into a per-element MSE.
- $1/24$ outside, in $\bar d_t^{(p)}$, averages those per-seed MSEs across the 24 repeated seeds for that pair.

So they are not redundant.

For a toy example, I’ll use:
- $4$ groups,
- $2$ pairs per group,
- $3$ seeds per pair,
- latent size $N=4$.

In the real study the pair-level average uses $24$; in this toy example we replace that by $3$.

## 1. The equations in the toy setting

For one pair $p$ and one seed $s$,
$$
d_t^{(p,s)}=\frac{1}{N}\left\|z_t^{\mathrm{mono},(p,s)}-z_t^{\mathrm{poe},(p,s)}\right\|_2^2
$$
with $N=4$.

The terminal version is just the same quantity at the final denoising step:
$$
d_T^{(p,s)}=d_t^{(p,s)}\big|_{t=T}.
$$

Since the toy has $3$ seeds, the pair-level summaries become
$$
\bar d_t^{(p)}=\frac{1}{3}\sum_{s=1}^{3} d_t^{(p,s)}, \qquad
\bar d_T^{(p)}=\frac{1}{3}\sum_{s=1}^{3} d_T^{(p,s)}.
$$

In the real study, that $3$ becomes $24$.

## 2. One record worked out fully

Take Group 1, Pair 1, Seed 1, at some intermediate step $t$.

Let
$$
z_t^{\mathrm{mono},(1,1)}-z_t^{\mathrm{poe},(1,1)}=[1,1,0,0].
$$

Then
$$
\left\|z_t^{\mathrm{mono},(1,1)}-z_t^{\mathrm{poe},(1,1)}\right\|_2^2
=1^2+1^2+0^2+0^2=2.
$$

So
$$
d_t^{(1,1)}=\frac{1}{4}\cdot 2=0.50.
$$

Now at the terminal step $T$, let
$$
z_T^{\mathrm{mono},(1,1)}-z_T^{\mathrm{poe},(1,1)}=[1,1,1,1].
$$

Then
$$
\left\|z_T^{\mathrm{mono},(1,1)}-z_T^{\mathrm{poe},(1,1)}\right\|_2^2
=1+1+1+1=4,
$$
so
$$
d_T^{(1,1)}=\frac{1}{4}\cdot 4=1.00.
$$

That is all $d_T^{(p,s)}$ means: same formula, but evaluated at the last step.

## 3. One pair worked out across all 3 seeds

Still in Group 1, Pair 1.

### At step $t$
Seed 1:
$$
[1,1,0,0] \Rightarrow \| \cdot \|_2^2=2 \Rightarrow d_t^{(1,1)}=2/4=0.50
$$

Seed 2:
$$
[1,1,1,0] \Rightarrow \| \cdot \|_2^2=3 \Rightarrow d_t^{(1,2)}=3/4=0.75
$$

Seed 3:
$$
[2,0,0,0] \Rightarrow \| \cdot \|_2^2=4 \Rightarrow d_t^{(1,3)}=4/4=1.00
$$

Now average over the $3$ seeds:
$$
\bar d_t^{(1)}=\frac{1}{3}\left(0.50+0.75+1.00\right)=\frac{2.25}{3}=0.75.
$$

Equivalent expanded form:
$$
\bar d_t^{(1)}
=\frac{1}{3}\sum_{s=1}^3 d_t^{(1,s)}
=\frac{1}{3}\sum_{s=1}^3 \frac{1}{4}\|\Delta_t^{(1,s)}\|_2^2
=\frac{1}{12}(2+3+4)=\frac{9}{12}=0.75.
$$

This is the important point:
- $1/4$ averages over latent elements,
- $1/3$ averages over seeds,
- together they become $1/(3\cdot 4)=1/12$.

### At step $T$
Seed 1:
$$
[1,1,1,1] \Rightarrow d_T^{(1,1)}=4/4=1.00
$$

Seed 2:
$$
[1,1,0,0] \Rightarrow d_T^{(1,2)}=2/4=0.50
$$

Seed 3:
$$
[2,1,0,0] \Rightarrow d_T^{(1,3)}=5/4=1.25
$$

Then
$$
\bar d_T^{(1)}=\frac{1}{3}(1.00+0.50+1.25)=\frac{2.75}{3}\approx 0.917.
$$

## 4. Full toy dataset

Below I choose arbitrary valid MSE values for all groups. Since $N=4$, values in steps of $0.25$ are easy to interpret.

| Group | Pair | \(d_t^{(p,1)}\) | \(d_t^{(p,2)}\) | \(d_t^{(p,3)}\) | \(\bar d_t^{(p)}\) | \(d_T^{(p,1)}\) | \(d_T^{(p,2)}\) | \(d_T^{(p,3)}\) | \(\bar d_T^{(p)}\) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| G1 | \(p_1\) | 0.50 | 0.75 | 1.00 | 0.75 | 1.00 | 0.50 | 1.25 | 0.917 |
| G1 | \(p_2\) | 0.25 | 0.50 | 0.25 | 0.333 | 0.50 | 0.75 | 0.50 | 0.583 |
| G2 | \(p_1\) | 0.75 | 1.00 | 1.25 | 1.00 | 1.00 | 1.25 | 1.50 | 1.25 |
| G2 | \(p_2\) | 0.50 | 0.75 | 1.00 | 0.75 | 0.75 | 1.00 | 1.25 | 1.00 |
| G3 | \(p_1\) | 1.25 | 1.50 | 1.75 | 1.50 | 1.75 | 2.00 | 2.25 | 2.00 |
| G3 | \(p_2\) | 1.00 | 1.25 | 1.50 | 1.25 | 1.50 | 1.75 | 2.00 | 1.75 |
| G4 | \(p_1\) | 2.00 | 2.25 | 2.50 | 2.25 | 2.50 | 2.75 | 3.00 | 2.75 |
| G4 | \(p_2\) | 1.75 | 2.00 | 2.25 | 2.00 | 2.25 | 2.50 | 2.75 | 2.50 |

## 5. What “taxonomy is the unit of inference” means

In this toy example:
- each group has $2 × 3 = 6$ pair-seed records,
- but only $2$ pair-level summaries.

So for Group 1, the terminal quantities used for group-wise comparison are not all 6 seed values:
$$
\{1.00,0.50,1.25,0.50,0.75,0.50\}.
$$

Instead, they are the 2 pair summaries:
$$
\{\bar d_T^{(p_1)}, \bar d_T^{(p_2)}\} = \{0.917,\;0.583\}.
$$

Likewise:
- G2 uses \(\{1.25, 1.00\}\)
- G3 uses \(\{2.00, 1.75\}\)
- G4 uses \(\{2.75, 2.50\}\)

That is what the sentence means:
- first consolidate seed-level measurements within each pair,
- then compare groups using the pair summaries.

## 6. Why the two averages are both needed

For one pair and one seed:
$$
d_t^{(p,s)}=\frac{1}{N}\sum_{i=1}^N (\Delta_{t,i}^{(p,s)})^2
$$
is an average over latent coordinates.

For one pair across all seeds:
$$
\bar d_t^{(p)}=\frac{1}{S}\sum_{s=1}^S d_t^{(p,s)}
$$
is an average over repeated runs.

Combining them:
$$
\bar d_t^{(p)}
=
\frac{1}{S}\sum_{s=1}^S \left(\frac{1}{N}\sum_{i=1}^N (\Delta_{t,i}^{(p,s)})^2\right)
=
\frac{1}{SN}\sum_{s=1}^S\sum_{i=1}^N (\Delta_{t,i}^{(p,s)})^2.
$$

So in the real study:
$$
\bar d_t^{(p)}=\frac{1}{24N}\sum_{s=1}^{24}\sum_{i=1}^{N}(\Delta_{t,i}^{(p,s)})^2.
$$

That is mathematically exactly what you want:
- normalize within each latent,
- then average across the 24 seeds.

## 7. Final interpretation

Using the toy setup:
- $d_t^{(p,s)}$ is one scalar for one pair and one seed at one denoising step.
- $d_T^{(p,s)}$ is the same scalar at the final step.
- $\bar d_t^{(p)}$ and $\bar d_T^{(p)}$ average those seed-level values for a fixed pair.
- group comparison is then done using the pair-level summaries, not the raw seed records.

In [ ]:
from statistics import mean


class SVGFigure:
    def __init__(self, svg):
        self.svg = svg

    def _repr_svg_(self):
        return self.svg

    def __repr__(self):
        return self.svg


def squared_l2(vec):
    return sum(v * v for v in vec)


def mean_of(values):
    return sum(values) / len(values)


def fmt(value):
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def print_table(rows, headers):
    widths = {}
    for h in headers:
        widths[h] = max(len(h), max(len(fmt(row[h])) for row in rows))
    header_line = ' | '.join(h.ljust(widths[h]) for h in headers)
    rule_line = '-+-'.join('-' * widths[h] for h in headers)
    print(header_line)
    print(rule_line)
    for row in rows:
        print(' | '.join(fmt(row[h]).ljust(widths[h]) for h in headers))


def compute_pair_summary(records, groups, pairs):
    rows = []
    for group in groups:
        for pair in pairs:
            subset = [r for r in records if r['group'] == group and r['pair'] == pair]
            rows.append({
                'group': group,
                'pair': pair,
                'd_t_bar': mean_of([r['d_t'] for r in subset]),
                'd_T_bar': mean_of([r['d_T'] for r in subset]),
            })
    return rows


def compute_group_summary(pair_summary, groups):
    rows = []
    for group in groups:
        subset = [r for r in pair_summary if r['group'] == group]
        rows.append({
            'group': group,
            'mean_pair_d_t': mean_of([r['d_t_bar'] for r in subset]),
            'mean_pair_d_T': mean_of([r['d_T_bar'] for r in subset]),
        })
    return rows


def svg_wrap(width, height, body, title=''):
    title_text = f"<text x='{width/2}' y='24' text-anchor='middle' font-size='18' font-family='sans-serif'>{title}</text>" if title else ''
    return SVGFigure(
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>"
        f"<rect width='100%' height='100%' fill='white'/>"
        f"{title_text}{body}</svg>"
    )


def single_record_svg(values, avg_value, title):
    width, height = 700, 380
    left, right, top, bottom = 70, 30, 50, 60
    plot_w = width - left - right
    plot_h = height - top - bottom
    ymax = max(max(values), avg_value) * 1.25
    body = []
    body.append(f"<line x1='{left}' y1='{height-bottom}' x2='{width-right}' y2='{height-bottom}' stroke='black'/>" )
    body.append(f"<line x1='{left}' y1='{top}' x2='{left}' y2='{height-bottom}' stroke='black'/>" )
    for i, v in enumerate(values, start=1):
        x = left + (i - 0.75) * (plot_w / len(values))
        bw = (plot_w / len(values)) * 0.5
        bh = (v / ymax) * plot_h
        y = height - bottom - bh
        body.append(f"<rect x='{x}' y='{y}' width='{bw}' height='{bh}' fill='#4E79A7' opacity='0.85'/>" )
        body.append(f"<text x='{x + bw/2}' y='{height-bottom+20}' text-anchor='middle' font-size='12' font-family='sans-serif'>{i}</text>" )
        body.append(f"<text x='{x + bw/2}' y='{y-6}' text-anchor='middle' font-size='12' font-family='sans-serif'>{v:.0f}</text>" )
    y_avg = height - bottom - (avg_value / ymax) * plot_h
    body.append(f"<line x1='{left}' y1='{y_avg}' x2='{width-right}' y2='{y_avg}' stroke='#E15759' stroke-width='2' stroke-dasharray='6,4'/>" )
    body.append(f"<text x='{width-right-5}' y='{y_avg-6}' text-anchor='end' font-size='13' font-family='sans-serif' fill='#E15759'>Mean = {avg_value:.2f}</text>" )
    body.append(f"<text x='{width/2}' y='{height-18}' text-anchor='middle' font-size='13' font-family='sans-serif'>Latent element index</text>" )
    body.append(f"<text x='18' y='{height/2}' transform='rotate(-90 18 {height/2})' text-anchor='middle' font-size='13' font-family='sans-serif'>Squared difference</text>" )
    return svg_wrap(width, height, ''.join(body), title)


def seed_vs_pair_svg(records, pair_summary, groups, pairs):
    width, height = 980, 760
    panel_w, panel_h = 430, 290
    x_offsets = [40, 510, 40, 510]
    y_offsets = [50, 50, 380, 380]
    ymax = max(r['d_T'] for r in records) * 1.15
    body = []
    for idx, group in enumerate(groups):
        px, py = x_offsets[idx], y_offsets[idx]
        left, right, top, bottom = px + 55, px + panel_w - 20, py + 30, py + panel_h - 45
        body.append(f"<rect x='{px}' y='{py}' width='{panel_w}' height='{panel_h}' fill='none' stroke='#cccccc'/>" )
        body.append(f"<text x='{px + panel_w/2}' y='{py + 22}' text-anchor='middle' font-size='16' font-family='sans-serif'>{group}</text>" )
        body.append(f"<line x1='{left}' y1='{bottom}' x2='{right}' y2='{bottom}' stroke='black'/>" )
        body.append(f"<line x1='{left}' y1='{top}' x2='{left}' y2='{bottom}' stroke='black'/>" )
        for pair_idx, pair in enumerate(pairs, start=1):
            x_center = left + pair_idx * (right - left) / 3
            subset = [r for r in records if r['group'] == group and r['pair'] == pair]
            jitters = [-12, 0, 12]
            for jitter, row in zip(jitters, subset):
                y = bottom - (row['d_T'] / ymax) * (bottom - top)
                body.append(f"<circle cx='{x_center + jitter}' cy='{y}' r='5' fill='#4E79A7' opacity='0.8'/>" )
            mean_val = [r['d_T_bar'] for r in pair_summary if r['group'] == group and r['pair'] == pair][0]
            y_mean = bottom - (mean_val / ymax) * (bottom - top)
            body.append(f"<line x1='{x_center - 18}' y1='{y_mean}' x2='{x_center + 18}' y2='{y_mean}' stroke='black' stroke-width='3'/>" )
            body.append(f"<text x='{x_center}' y='{y_mean - 8}' text-anchor='middle' font-size='11' font-family='sans-serif'>{mean_val:.3f}</text>" )
            body.append(f"<text x='{x_center}' y='{bottom + 20}' text-anchor='middle' font-size='12' font-family='sans-serif'>{pair}</text>" )
    return svg_wrap(width, height, ''.join(body), 'Seed-level terminal gaps and pair-level means')


def pair_bar_svg(pair_summary, groups, pairs):
    width, height = 860, 380
    left, right, top, bottom = 70, 20, 50, 70
    plot_w = width - left - right
    plot_h = height - top - bottom
    rows = []
    for group in groups:
        for pair in pairs:
            rows.append(next(r for r in pair_summary if r['group'] == group and r['pair'] == pair))
    ymax = max(r['d_T_bar'] for r in rows) * 1.15
    colors = {'G1': '#59A14F', 'G2': '#76B7B2', 'G3': '#F28E2B', 'G4': '#E15759'}
    body = []
    body.append(f"<line x1='{left}' y1='{height-bottom}' x2='{width-right}' y2='{height-bottom}' stroke='black'/>" )
    body.append(f"<line x1='{left}' y1='{top}' x2='{left}' y2='{height-bottom}' stroke='black'/>" )
    slot = plot_w / len(rows)
    for i, row in enumerate(rows):
        x = left + i * slot + slot * 0.15
        bw = slot * 0.7
        bh = (row['d_T_bar'] / ymax) * plot_h
        y = height - bottom - bh
        body.append(f"<rect x='{x}' y='{y}' width='{bw}' height='{bh}' fill='{colors[row['group']]}' opacity='0.9'/>" )
        body.append(f"<text x='{x + bw/2}' y='{height-bottom+18}' text-anchor='middle' font-size='11' font-family='sans-serif'>{row['group']}-{row['pair']}</text>" )
        body.append(f"<text x='{x + bw/2}' y='{y-6}' text-anchor='middle' font-size='11' font-family='sans-serif'>{row['d_T_bar']:.3f}</text>" )
    body.append(f"<text x='{width/2}' y='{height-18}' text-anchor='middle' font-size='13' font-family='sans-serif'>Pair-level summaries</text>" )
    body.append(f"<text x='18' y='{height/2}' transform='rotate(-90 18 {height/2})' text-anchor='middle' font-size='13' font-family='sans-serif'>Pair-level terminal gap d̄_T^(p)</text>" )
    return svg_wrap(width, height, ''.join(body), 'Pair-level summaries used for group-wise comparison')


def dual_heatmap_svg(pair_summary, groups, pairs):
    width, height = 980, 360
    body = []
    panels = [
        ('d_t_bar', 'Pair-level d̄_t^(p)', 40, '#4E79A7'),
        ('d_T_bar', 'Pair-level d̄_T^(p)', 510, '#E15759'),
    ]
    for key, title, x0, color in panels:
        y0 = 50
        cell_w, cell_h = 150, 55
        vals = [r[key] for r in pair_summary]
        vmax = max(vals)
        body.append(f"<text x='{x0 + 160}' y='28' text-anchor='middle' font-size='16' font-family='sans-serif'>{title}</text>" )
        for i, group in enumerate(groups):
            body.append(f"<text x='{x0 - 10}' y='{y0 + i*cell_h + 35}' text-anchor='end' font-size='12' font-family='sans-serif'>{group}</text>" )
            for j, pair in enumerate(pairs):
                val = next(r[key] for r in pair_summary if r['group'] == group and r['pair'] == pair)
                opacity = 0.15 + 0.85 * (val / vmax)
                x = x0 + j * cell_w
                y = y0 + i * cell_h
                body.append(f"<rect x='{x}' y='{y}' width='{cell_w}' height='{cell_h}' fill='{color}' opacity='{opacity:.3f}' stroke='white'/>" )
                body.append(f"<text x='{x + cell_w/2}' y='{y + 32}' text-anchor='middle' font-size='12' font-family='sans-serif' fill='black'>{val:.3f}</text>" )
        for j, pair in enumerate(pairs):
            body.append(f"<text x='{x0 + j*cell_w + cell_w/2}' y='{y0 + len(groups)*cell_h + 20}' text-anchor='middle' font-size='12' font-family='sans-serif'>{pair}</text>" )
    return svg_wrap(width, height, ''.join(body), 'Heatmap of pair-level summaries')


N = 4
S = 3
groups = ['G1', 'G2', 'G3', 'G4']
pairs = ['p1', 'p2']
seeds = [1, 2, 3]

print(f'N = {N} latent elements')
print(f'S = {S} seeds per pair in the toy example')
print(f'Groups = {groups}')
print(f'Pairs per group = {pairs}')

In [ ]:
delta_t = [1.0, 1.0, 0.0, 0.0]
delta_T = [1.0, 1.0, 1.0, 1.0]

squared_norm_t = squared_l2(delta_t)
d_t_11 = squared_norm_t / N

squared_norm_T = squared_l2(delta_T)
d_T_11 = squared_norm_T / N

print('Intermediate step example:')
print('delta_t =', delta_t)
print('||delta_t||_2^2 =', squared_norm_t)
print('d_t^(1,1) = (1/N) * ||delta_t||_2^2 =', d_t_11)
print()
print('Terminal step example:')
print('delta_T =', delta_T)
print('||delta_T||_2^2 =', squared_norm_T)
print('d_T^(1,1) = (1/N) * ||delta_T||_2^2 =', d_T_11)

In [ ]:
delta_t_seeds = {
    1: [1.0, 1.0, 0.0, 0.0],
    2: [1.0, 1.0, 1.0, 0.0],
    3: [2.0, 0.0, 0.0, 0.0],
}

delta_T_seeds = {
    1: [1.0, 1.0, 1.0, 1.0],
    2: [1.0, 1.0, 0.0, 0.0],
    3: [2.0, 1.0, 0.0, 0.0],
}

d_t_values = {s: squared_l2(delta_t_seeds[s]) / N for s in seeds}
d_T_values = {s: squared_l2(delta_T_seeds[s]) / N for s in seeds}

d_t_bar = mean_of(list(d_t_values.values()))
d_T_bar = mean_of(list(d_T_values.values()))
expanded_d_t_bar = sum(squared_l2(delta_t_seeds[s]) for s in seeds) / (S * N)

print('Per-seed d_t values:')
for s in seeds:
    print(f'  seed {s}: delta = {delta_t_seeds[s]}, d_t^(1,{s}) = {d_t_values[s]:.3f}')
print(f'Pair-level average d̄_t^(1) = (1/3) * sum_s d_t^(1,s) = {d_t_bar:.3f}')
print(f'Expanded form d̄_t^(1) = (1/(3*4)) * sum squared differences = {expanded_d_t_bar:.3f}')
print()
print('Per-seed d_T values:')
for s in seeds:
    print(f'  seed {s}: delta = {delta_T_seeds[s]}, d_T^(1,{s}) = {d_T_values[s]:.3f}')
print(f'Pair-level average d̄_T^(1) = (1/3) * sum_s d_T^(1,s) = {d_T_bar:.3f}')

In [ ]:
toy_values = {
    ('G1', 'p1'): {'d_t': [0.50, 0.75, 1.00], 'd_T': [1.00, 0.50, 1.25]},
    ('G1', 'p2'): {'d_t': [0.25, 0.50, 0.25], 'd_T': [0.50, 0.75, 0.50]},
    ('G2', 'p1'): {'d_t': [0.75, 1.00, 1.25], 'd_T': [1.00, 1.25, 1.50]},
    ('G2', 'p2'): {'d_t': [0.50, 0.75, 1.00], 'd_T': [0.75, 1.00, 1.25]},
    ('G3', 'p1'): {'d_t': [1.25, 1.50, 1.75], 'd_T': [1.75, 2.00, 2.25]},
    ('G3', 'p2'): {'d_t': [1.00, 1.25, 1.50], 'd_T': [1.50, 1.75, 2.00]},
    ('G4', 'p1'): {'d_t': [2.00, 2.25, 2.50], 'd_T': [2.50, 2.75, 3.00]},
    ('G4', 'p2'): {'d_t': [1.75, 2.00, 2.25], 'd_T': [2.25, 2.50, 2.75]},
}

records = []
for (group, pair), vals in toy_values.items():
    for idx, seed in enumerate(seeds):
        records.append({
            'group': group,
            'pair': pair,
            'seed': seed,
            'd_t': vals['d_t'][idx],
            'd_T': vals['d_T'][idx],
        })

pair_summary = compute_pair_summary(records, groups, pairs)

print('Full toy dataset (seed-level records):')
print_table(records, ['group', 'pair', 'seed', 'd_t', 'd_T'])
print()
print('Pair-level summaries:')
print_table(pair_summary, ['group', 'pair', 'd_t_bar', 'd_T_bar'])

In [ ]:
group_summary = compute_group_summary(pair_summary, groups)

print('Group-level summaries from pair-level quantities:')
print_table(group_summary, ['group', 'mean_pair_d_t', 'mean_pair_d_T'])
print()
print('Example of what “taxonomy is the unit of inference” means:')
for group in groups:
    vals = [r['d_T_bar'] for r in pair_summary if r['group'] == group]
    print(f'  {group}: use pair summaries {vals}, not all raw seed values directly')

## Visualizations

The following cells return inline SVG figures using only the Python standard library.

In [ ]:
single_record_svg([v * v for v in delta_t], d_t_11, 'Single pair-seed record: averaging over latent elements (1/N)')

In [ ]:
seed_vs_pair_svg(records, pair_summary, groups, pairs)

In [ ]:
pair_bar_svg(pair_summary, groups, pairs)

In [ ]:
dual_heatmap_svg(pair_summary, groups, pairs)